In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.special import softmax
from src import config as C
from src.calibration import fit_temperature, ece, brier
from src.selective import *

d = np.load(C.PROC / 'oof_ds1_v5_augwd.npz')
L = d['logits']                    # [seed, n, 4]
y, rec = d['y'], d['rec']
Lens = L.mean(0)                   # ensemble by logit averaging
pred = Lens.argmax(1); correct = pred == y
print('ensemble acc', correct.mean().round(3))

In [ ]:
T = fit_temperature(Lens, y)
P_raw, P_cal = softmax(Lens, 1), softmax(Lens / T, 1)
print('T =', round(T, 3))
for name, P in [('raw', P_raw), ('temp-scaled', P_cal)]:
    print(f'{name:12s} ECE {ece(P, y):.4f} | Brier {brier(P, y):.4f}')

In [ ]:
scores = {'MSP raw': P_raw.max(1), 'MSP calibrated': P_cal.max(1),
          'neg. entropy (cal.)': (P_cal * np.log(P_cal + 1e-12)).sum(1)}
levels = [1.0, 0.95, 0.9, 0.8, 0.7]
base_risk = 1 - correct.mean()
rows = []
for name, s in scores.items():
    rows.append([name, round(aurc(s, correct), 4)] +
                [round(selective_risk(accept_at_coverage(s, c), correct), 4) for c in levels])
rows.append(['random', round(base_risk, 4)] + [round(base_risk, 4)] * len(levels))
rows.append(['oracle', round(aurc(correct.astype(float), correct), 4)] +
            [round(selective_risk(oracle_accept(correct, c), correct), 4) for c in levels])
print(pd.DataFrame(rows, columns=['method', 'AURC'] + [f'risk@{int(c*100)}%' for c in levels]).to_string(index=False))

In [ ]:
s = scores['MSP calibrated']
out = []
for c in levels[1:]:
    cc = class_coverage(accept_at_coverage(s, c), y)
    out.append([f'{int(c*100)}%', *cc.round(3), round(np.nanmax(cc[:3]) - np.nanmin(cc[:3]), 3)])
print(pd.DataFrame(out, columns=['coverage'] + C.CLASSES + ['gap N/S/V']).to_string(index=False))
print('F is descriptive only (one dominant record).')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
for name, s_ in scores.items():
    cov, risk = risk_coverage(s_, correct); ax.plot(cov, risk, label=name)
cov, risk = risk_coverage(correct.astype(float), correct); ax.plot(cov, risk, 'k--', label='oracle')
ax.axhline(base_risk, color='grey', ls=':', label='random')
ax.set_xlabel('coverage'); ax.set_ylabel('selective risk'); ax.legend(fontsize=7); plt.tight_layout()